Imports

In [1]:
# import of required libraries

from esdl import esdl
from esdl.esdl_handler import EnergySystemHandler
import pandas as pd
import numpy as np
import numpy_financial as npf
from decimal import Decimal, ROUND_HALF_UP 
import matplotlib.pyplot as plt


In [2]:
# Import the business case runner
from set_up.generic_business_case_runner import *

In [3]:
# This imports all input variables from EL_input_data.py
from assets.OT_input_data import *

# import all data from ESDL
from esdl_files.NSE_get_data_from_ESDL import *


Active Scenario: most_likely (Index: 1)
loading cached data from profiles.pkl...
All variables saved to NSE_get_data_from_ESDL.pkl


In [4]:
asset_parameters
# Note: 24.9MW is the required natural gas boiler capacity

name,power,efficiency,investment_costs,fixed_opex,variable_opex,wacc
TNVDW,700000000.0,,1750.0,2.25,5.0,8.5
Electrolyzer,500.0,0.6,2000.0,2.0,0.0,8.25
Offtaker,24900000.0,,0.0,0.0,0.0,10.5


Set up calendar_year, business_case_year, operations_years, and decommissioning_years lists

In [5]:
from set_up.construct_timelines import (
    construct_calendar_year_list,
    construct_business_case_year_list,
    construct_operations_years_list,
    construct_decommissioning_years_list
)

Start business case analysis

Construction Phase

In [6]:
def construction_phase(**kwargs):
    ''' This function creates a dataframe that contains all cashflows in the construction phase'''

    # 1. Get the parameters we need
    duration_construction = kwargs["duration_construction"]
    capex_h2_station = kwargs["capex_h2_receiving_station"]
    capex_h2_pipeline = kwargs["capex_h2_pipeline"]
    capex_h2_boiler = kwargs["capex_h2_boiler"]
    capex_ng_boiler = kwargs["capex_ng_boiler"]

    # 2. Get the timelines
    calendar_years = construct_calendar_year_list(**kwargs)

    # 3. Construct df
    row_names = [
        "capex_h2_boiler",
        "capex_h2_station",
        "capex_h2_pipeline",
        "total_cash_outflow_investment",
        "avoided_capex_ng_boiler",
        "total_avoided_costs",
        "total_cashflow_investment",
    ]
    df_construction_phase = pd.DataFrame(0.0, index=row_names, columns=calendar_years)
    df_construction_phase.columns.name = "construction_phase"

    # CASH OUTFLOWS

    # 4. Calculate yearly CAPEX values
    yearly_capex_h2_boiler = -capex_h2_boiler / duration_construction
    yearly_capex_h2_station = -capex_h2_station / duration_construction
    yearly_capex_h2_pipeline = -capex_h2_pipeline / duration_construction

    # 5. Add CAPEX numbers to construction years
    df_construction_phase.loc['capex_h2_boiler'] = np.where(
        df_construction_phase.columns.isin(construction_years_list),
        yearly_capex_h2_boiler, 0.0
    )

    df_construction_phase.loc['capex_h2_station'] = np.where(
        df_construction_phase.columns.isin(construction_years_list),
        yearly_capex_h2_station, 0.0
    )

    df_construction_phase.loc['capex_h2_pipeline'] = np.where(
        df_construction_phase.columns.isin(construction_years_list),
        yearly_capex_h2_pipeline, 0.0
    )

        
    # 6. Calculate total investments   
    total_cash_outflow_investment = (
        df_construction_phase.loc['capex_h2_station'] 
        + df_construction_phase.loc['capex_h2_pipeline'] 
        + df_construction_phase.loc['capex_h2_boiler'])
    df_construction_phase.loc['total_cash_outflow_investment'] = total_cash_outflow_investment
    

    # AVOIDED COSTS

    # 7. Calculate yearly CAPEX values
    yearly_capex_ng_boiler = capex_ng_boiler / duration_construction

    # 8. Add CAPEX numbers to construction years
    df_construction_phase.loc['avoided_capex_ng_boiler'] = np.where(
        df_construction_phase.columns.isin(construction_years_list),
        yearly_capex_ng_boiler, 0.0
    )


    # 9. Calculate total avoided costs and add to df
    total_avoided_costs = df_construction_phase.loc['avoided_capex_ng_boiler']
    df_construction_phase.loc['total_avoided_costs'] = total_avoided_costs

    # 10. Calculate total cashflow from investments then add to df
    total_cashflow_investment = np.array(total_cash_outflow_investment) + np.array(total_avoided_costs)
    df_construction_phase.loc['total_cashflow_investment'] = total_cashflow_investment

    return df_construction_phase

In [7]:
df_construction_phase = construction_phase(**ot_parameters)
df_construction_phase.style.format(precision=2)

construction_phase,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
capex_h2_boiler,0.00,-3.78,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
capex_h2_station,0.00,-1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
capex_h2_pipeline,0.00,-10.81,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
total_cash_outflow_investment,0.00,-15.59,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
avoided_capex_ng_boiler,0.00,1.72,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
total_avoided_costs,0.00,1.72,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
total_cashflow_investment,0.00,-13.87,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


Operational Phase

In [8]:
def operational_phase(**kwargs):
    ''' This function creates a dataframe that contains all cashflows in the operational phase'''

    # 1. Get the parameters we need
    inflation = kwargs["inflation"]
    h2_boiler_opex = kwargs["h2_boiler_opex"]
    network_costs_h2 = kwargs["network_costs_h2"]
    h2_price = kwargs["h2_price"]
    hwi_price = kwargs["hwi_price"]

    ng_boiler_opex = kwargs["ng_boiler_opex"]
    network_costs_ng = kwargs["network_costs_ng"]
    carbon_permits = kwargs["carbon_permits"]
    ng_price = kwargs["ng_price"]
    ng_tax_costs = kwargs["ng_tax_costs"]

    # 2. Get the timelines
    calendar_years = construct_calendar_year_list(**kwargs)
    operational_years = construct_operations_years_list(**kwargs)
    business_case_years = construct_business_case_year_list(**kwargs)

    # 3. Construct df
    row_names = ["h2_boiler_opex", "network_costs_h2", "purchasing_h2",
        "purchasing_hwi", "total_outflow_opex", "avoided_ng_boiler_opex",
        "network_costs_ng", "eu_carbon_permits", "ng_costs",
        "ng_tax_costs", "total_avoided_costs", "net_cashflow_operations"]
    
    df_operational_phase = pd.DataFrame(0.0, index=row_names, columns=calendar_years)
    df_operational_phase.columns.name = "operational_phase"
    
    # 4. Calculate inflation factors array 
    inf_factors = (1 + inflation) ** business_case_years
    # Turn it into a series to give the calendar years as index
    # This way we can use it in our loop over the operational years
    inf_factor_series = pd.Series(inf_factors, index=calendar_years)


    # 5. Calculate the cashflows
    
    for x in operational_years:

        # Get the inflation factor for that year
        inf_factor = inf_factor_series[x]

        # CASH OUTFLOWS
        df_operational_phase.loc["h2_boiler_opex", x] = (
            -h2_boiler_opex.loc["opex_h2_boiler", x] * inf_factor)
        
        df_operational_phase.loc["network_costs_h2", x] = (
            -network_costs_h2 * inf_factor)

        df_operational_phase.loc["purchasing_h2", x] = (
            -(h2_price.loc["ot_h2_price", x] * ot_h2_demand_mwh / 1E6) * inf_factor)

        df_operational_phase.loc["purchasing_hwi", x] = (
            -(hwi_price.loc["ot_hwi_costs", x] *ot_hwi_percentage.loc['hwi_percentage',x] 
              * inf_factor))


        # AVOIDED COSTS
        df_operational_phase.loc["avoided_ng_boiler_opex", x] = (
            ng_boiler_opex.loc["opex_ng_boiler", x] * inf_factor)
        
        df_operational_phase.loc["network_costs_ng", x] = (
            network_costs_ng * inf_factor)
        
        df_operational_phase.loc["eu_carbon_permits", x] = (
            (co2_per_mwh_ng * ot_ng_demand_mwh / 1000 * carbon_permits.loc["carbon_permits", x] / 1e6)
            * inf_factor)

        df_operational_phase.loc["ng_costs", x] = (
            (ng_price.loc["ot_ng_price", x] * ot_ng_demand_mwh / 1e6) * inf_factor)

        df_operational_phase.loc["ng_tax_costs", x] = (
            (ng_tax_costs * ot_ng_demand_m3 / 1e6) * inf_factor)


    # 6. Calculate  outflow totals 
    outflow_rows = [
        "h2_boiler_opex", 
        "network_costs_h2",
        "purchasing_h2", 
        # "purchasing_hwi",  # Hernieuwbare waterstofeenheid - not included in total for now because of uncertainty
    ]
    df_operational_phase.loc["total_outflow_opex"] = df_operational_phase.loc[outflow_rows].sum()

    # 7. Calculate avoided cost totals
    avoided_rows = [
        "avoided_ng_boiler_opex", "network_costs_ng", "eu_carbon_permits",
        "ng_costs", "ng_tax_costs",
    ]
    df_operational_phase.loc["total_avoided_costs"] = df_operational_phase.loc[avoided_rows].sum()

    # 9. Net cashflow from operation (=EBITDA)
    df_operational_phase.loc['net_cashflow_operations'] = (
        df_operational_phase.loc['total_outflow_opex'] + 
        df_operational_phase.loc['total_avoided_costs'])

    
    return df_operational_phase

In [9]:
df_operational_phase = operational_phase(**ot_parameters)
df_operational_phase.style.format(precision=2)

operational_phase,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
h2_boiler_opex,0.00,0.00,-0.12,-0.13,-0.13,-0.13,-0.13,-0.14,-0.14,-0.14,-0.15,-0.15,-0.15,-0.15,-0.16,-0.16,-0.16,-0.17,-0.17,-0.17,0.00
network_costs_h2,0.00,0.00,-8.08,-8.25,-8.41,-8.58,-8.75,-8.93,-9.10,-9.29,-9.47,-9.66,-9.85,-10.05,-10.25,-10.46,-10.67,-10.88,-11.10,-11.32,0.00
purchasing_h2,0.00,0.00,-14.52,-14.88,-15.26,-15.64,-16.04,-16.44,-16.85,-17.28,-17.71,-18.15,-17.86,-17.56,-17.23,-16.88,-16.52,-16.13,-15.71,-15.28,0.00
purchasing_hwi,0.00,0.00,-14.74,-15.03,-15.33,-15.64,-22.79,-23.25,-23.71,-24.18,-24.67,-25.16,-25.66,-26.18,-26.70,-27.24,-27.78,-28.34,-28.90,-29.48,0.00
total_outflow_opex,0.00,0.00,-22.72,-23.26,-23.80,-24.35,-24.92,-25.50,-26.10,-26.71,-27.33,-27.96,-27.87,-27.76,-27.64,-27.50,-27.35,-27.17,-26.98,-26.77,0.00
avoided_ng_boiler_opex,0.00,0.00,0.07,0.07,0.08,0.08,0.08,0.08,0.08,0.08,0.08,0.09,0.09,0.09,0.09,0.09,0.10,0.10,0.10,0.10,0.00
network_costs_ng,0.00,0.00,2.07,2.11,2.15,2.19,2.24,2.28,2.33,2.38,2.42,2.47,2.52,2.57,2.62,2.68,2.73,2.78,2.84,2.90,0.00
eu_carbon_permits,0.00,0.00,4.72,5.04,5.38,5.72,6.07,6.44,6.82,7.21,7.62,8.04,10.53,13.12,15.81,18.60,21.50,24.50,27.62,30.85,0.00
ng_costs,0.00,0.00,8.76,8.92,9.08,9.25,9.42,9.59,9.76,9.94,10.12,10.30,10.47,10.65,10.82,11.00,11.19,11.37,11.56,11.76,0.00
ng_tax_costs,0.00,0.00,1.42,1.45,1.48,1.51,1.54,1.57,1.60,1.64,1.67,1.70,1.74,1.77,1.81,1.84,1.88,1.92,1.96,1.99,0.00


Decommissioning phase

In [10]:
def decommissioning_phase(**kwargs):
    """ This function creates a dataframe that contains all cashflows in the decommissioning phase.
        Note that the H2 receiving station and the H2 pipeline are not taken into account for the decommissioning (because the lifetime is much higher than bc?).
        Could be included --> DISCUSS and align with BH bc """

    # 1. Get the parameters we need
    lifetime_investment = kwargs["lifetime_investment"]
    inflation = kwargs["inflation"]
    duration_decommissioning = kwargs["duration_decommissioning"]
    capex_h2_boiler = kwargs["capex_h2_boiler"]
    capex_ng_boiler = kwargs["capex_ng_boiler"]
    h2_boiler_decommissioning_percentage = kwargs["h2_boiler_decommissioning_percentage"]
    ng_boiler_decommissioning_percentage = kwargs["ng_boiler_decommissioning_percentage"]

    # 2. Get the timelines
    calendar_years = construct_calendar_year_list(**kwargs)
    decommissioning_years = construct_decommissioning_years_list(**kwargs)
    first_decommissioning_year = decommissioning_years[0]
    business_case_years = construct_business_case_year_list(**kwargs)

    # 3. Construct df
    row_names = [
        "decommissioning_cost",
        "decommissioning_avoided_cost",
        "total_cashflow_decommissioning",
    ]
    df_decommissioning_phase = pd.DataFrame(0.0, index=row_names, columns=calendar_years)
    df_decommissioning_phase.columns.name = "decommissioning_phase"

    # 4. Calculate inflation factors indexed by calendar year
    inf_factors = (1 + inflation) ** business_case_years
    # Turn it into a series to give the calendar years as index
    # This way we can use it in our loop over the operational years
    inf_factor_series = pd.Series(inf_factors, index=calendar_years)

    # 5. Calculate yearly decommissioning amounts
    yearly_decom_h2_boiler = (
        -capex_h2_boiler * h2_boiler_decommissioning_percentage / duration_decommissioning)
    
    yearly_decom_ng_boiler = (
        capex_ng_boiler * ng_boiler_decommissioning_percentage / duration_decommissioning)

    # 6. Calculate decommissioning costs
    for x in decommissioning_years:
        inf_factor = inf_factor_series[x]
        df_decommissioning_phase.loc["decommissioning_cost", x] = yearly_decom_h2_boiler * inf_factor

    # 7. Calculate avoided decommissioning costs
    for x in decommissioning_years:
        inf_factor = inf_factor_series[x]
        df_decommissioning_phase.loc["decommissioning_avoided_cost", x] = yearly_decom_ng_boiler * inf_factor

    # Add Resale value of NG boiler because lifetie is longer than H2 boiler lifetime and business case length
    if lifetime_investment < ot_ng_boiler_depreciation: 
        yearly_depreciation = capex_ng_boiler / ot_ng_boiler_depreciation
        resale_value = capex_ng_boiler - (lifetime_investment * yearly_depreciation)

        # Substract the resale value, because by choosing H2 boiler, you don't have the NG resale value
        df_decommissioning_phase.loc['decommissioning_avoided_cost', first_decommissioning_year] -= resale_value


    # 7. Calculate totals
    df_decommissioning_phase.loc["total_cashflow_decommissioning"] = (
        df_decommissioning_phase.loc["decommissioning_cost"]
        + df_decommissioning_phase.loc["decommissioning_avoided_cost"]
    )

    return df_decommissioning_phase

In [11]:
df_decommissioning_phase = decommissioning_phase(**ot_parameters)
df_decommissioning_phase.style.format(precision=2)

decommissioning_phase,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
decommissioning_cost,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.11
decommissioning_avoided_cost,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.43
total_cashflow_decommissioning,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.54


Taxes and profits - part 1

In [12]:
from finances.taxes_debt_loans import taxes_and_profits_part1

df_taxes_and_profits_part1 = taxes_and_profits_part1(df_operational_phase,**ot_parameters)
df_taxes_and_profits_part1.style.format(precision=2)

taxes_and_profits,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
depreciation,0.00,0.00,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,0.00
ebit,0.00,0.00,-6.47,-6.45,-6.43,-6.40,-6.37,-6.34,-6.30,-6.26,-6.21,-6.16,-3.32,-0.36,2.72,5.92,9.25,12.71,16.30,20.03,0.00


Debt and loan - part 1

In [13]:
from finances.taxes_debt_loans import debt_and_loan_part1

df_debt_and_loan_part1 = debt_and_loan_part1(**ot_parameters)
df_debt_and_loan_part1.style.format(precision=2)

debt_and_loan,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
begin_of_year,0.00,0.00,10.41,9.99,9.55,9.07,8.57,8.02,7.44,6.82,6.16,5.45,4.68,3.87,3.00,2.07,1.07,0.00,0.00,0.00,0.00
drawdown,0.00,10.41,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
repayment_capital,0.00,0.00,-0.41,-0.44,-0.47,-0.51,-0.54,-0.58,-0.62,-0.66,-0.71,-0.76,-0.81,-0.87,-0.93,-1.00,-1.07,-0.00,-0.00,-0.00,0.00
end_of_year,0.00,10.41,9.99,9.55,9.07,8.57,8.02,7.44,6.82,6.16,5.45,4.68,3.87,3.00,2.07,1.07,0.00,0.00,0.00,0.00,0.00
repayment_interest,0.00,0.00,-0.73,-0.70,-0.67,-0.64,-0.60,-0.56,-0.52,-0.48,-0.43,-0.38,-0.33,-0.27,-0.21,-0.14,-0.07,-0.00,-0.00,-0.00,0.00


Taxes & profits - part 2

In [14]:
from finances.taxes_debt_loans import taxes_and_profits_part2

df_taxes_and_profits_part2 = taxes_and_profits_part2(df_operational_phase,
                                                     df_taxes_and_profits_part1,
                                                     df_debt_and_loan_part1,
                                                     **ot_parameters)
df_taxes_and_profits_part2.style.format(precision=2)

taxes_and_profits,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
depreciation,0.00,0.00,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,-0.80,0.00
ebit,0.00,0.00,-6.47,-6.45,-6.43,-6.40,-6.37,-6.34,-6.30,-6.26,-6.21,-6.16,-3.32,-0.36,2.72,5.92,9.25,12.71,16.30,20.03,0.00
interest_costs,0.00,0.00,-0.73,-0.70,-0.67,-0.64,-0.60,-0.56,-0.52,-0.48,-0.43,-0.38,-0.33,-0.27,-0.21,-0.14,-0.07,-0.00,-0.00,-0.00,0.00
ebt,0.00,0.00,-7.20,-7.15,-7.09,-7.03,-6.97,-6.90,-6.82,-6.73,-6.64,-6.54,-3.65,-0.63,2.51,5.77,9.17,12.71,16.30,20.03,0.00
tax_expenses,-0.00,-0.00,1.86,1.84,1.83,1.81,1.80,1.78,1.76,1.74,1.71,1.69,0.94,0.16,-0.65,-1.49,-2.37,-3.28,-4.21,-5.17,-0.00
net_profits,0.00,0.00,-5.34,-5.30,-5.26,-5.22,-5.17,-5.12,-5.06,-5.00,-4.93,-4.86,-2.71,-0.47,1.86,4.28,6.81,9.43,12.09,14.86,0.00


Debt & loan - part 2

In [15]:
from finances.taxes_debt_loans import debt_and_loan_part2

df_debt_and_loan_part2 = debt_and_loan_part2(df_operational_phase,
                                             df_debt_and_loan_part1,
                                             df_taxes_and_profits_part2,
                                             **ot_parameters)

df_debt_and_loan_part2.style.format(precision=2)

debt_and_loan,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
begin_of_year,0.00,0.00,10.41,9.99,9.55,9.07,8.57,8.02,7.44,6.82,6.16,5.45,4.68,3.87,3.00,2.07,1.07,0.00,0.00,0.00,0.00
drawdown,0.00,10.41,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
repayment_capital,0.00,0.00,-0.41,-0.44,-0.47,-0.51,-0.54,-0.58,-0.62,-0.66,-0.71,-0.76,-0.81,-0.87,-0.93,-1.00,-1.07,-0.00,-0.00,-0.00,0.00
end_of_year,0.00,10.41,9.99,9.55,9.07,8.57,8.02,7.44,6.82,6.16,5.45,4.68,3.87,3.00,2.07,1.07,0.00,0.00,0.00,0.00,0.00
repayment_interest,0.00,0.00,-0.73,-0.70,-0.67,-0.64,-0.60,-0.56,-0.52,-0.48,-0.43,-0.38,-0.33,-0.27,-0.21,-0.14,-0.07,-0.00,-0.00,-0.00,0.00
cash_flow_for_debt,0.00,0.00,-3.82,-3.81,-3.80,-3.79,-3.77,-3.76,-3.74,-3.72,-3.70,-3.68,-1.58,0.60,2.87,5.23,7.68,10.23,12.89,15.66,0.00
debt_service,0.00,0.00,-1.14,-1.14,-1.14,-1.14,-1.14,-1.14,-1.14,-1.14,-1.14,-1.14,-1.14,-1.14,-1.14,-1.14,-1.14,-0.00,-0.00,-0.00,0.00
cash_after_debt_service,0.00,0.00,-4.96,-4.95,-4.94,-4.93,-4.92,-4.90,-4.88,-4.86,-4.84,-4.82,-2.72,-0.54,1.73,4.08,6.54,10.23,12.89,15.66,0.00


Project reserves

In [16]:
from finances.project_reserves_and_equity import project_reserves

df_project_reserves = project_reserves(**ot_parameters)
df_project_reserves.style.format(precision=2)

project_reserves,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
contingency_injection,-1.39,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
contingency_reserve_balance,1.39,1.39,1.39,1.39,1.39,1.39,1.39,1.39,1.39,1.39,1.39,1.39,1.39,1.39,1.39,1.39,1.39,1.39,1.39,1.39,0.00
contingency_reserve_to_dividents,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.39


Equity funding

In [17]:
from finances.project_reserves_and_equity import equity_funding

df_equity_funding = equity_funding(df_decommissioning_phase,
               df_debt_and_loan_part2,
               df_construction_phase,
               **ot_parameters)

df_equity_funding.style.format(precision=2)

equity_funding,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
equity_injection,-1.39,-3.47,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.54
dividents_results,0.00,0.00,-4.96,-4.95,-4.94,-4.93,-4.92,-4.90,-4.88,-4.86,-4.84,-4.82,-2.72,-0.54,1.73,4.08,6.54,10.23,12.89,15.66,1.39
equity_cash_flow_result,-1.39,-3.47,-4.96,-4.95,-4.94,-4.93,-4.92,-4.90,-4.88,-4.86,-4.84,-4.82,-2.72,-0.54,1.73,4.08,6.54,10.23,12.89,15.66,0.85


Present value of cash flows

In [18]:
from finances.present_value_and_cumulative_cashflows import present_value_cashflows

df_present_value_cashflows = present_value_cashflows(df_construction_phase,
                                                     df_operational_phase,
                                                     df_decommissioning_phase,
                                                     df_equity_funding,
                                                     **ot_parameters)

df_present_value_cashflows.style.format(precision=2)

present_value,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
sum_net_project_cash_flows,0.00,-13.87,-5.67,-5.65,-5.63,-5.60,-5.57,-5.54,-5.50,-5.46,-5.41,-5.37,-2.52,0.44,3.51,6.72,10.04,13.50,17.10,20.83,-0.54
present_value_net_cashflows,0.00,-12.56,-4.65,-4.19,-3.78,-3.40,-3.06,-2.75,-2.47,-2.22,-2.00,-1.79,-0.76,0.12,0.87,1.50,2.03,2.47,2.83,3.12,-0.07
cumulative_value_net_cashflows,0.00,-12.56,-17.20,-21.39,-25.17,-28.57,-31.63,-34.38,-36.86,-39.08,-41.07,-42.86,-43.62,-43.50,-42.64,-41.13,-39.10,-36.63,-33.79,-30.67,-30.74
present_value_equity_cashflows,-1.39,-3.14,-4.06,-3.67,-3.31,-2.99,-2.70,-2.44,-2.20,-1.98,-1.78,-1.61,-0.82,-0.15,0.43,0.91,1.32,1.87,2.14,2.35,0.11


Cumulative equity and debt cashflows

In [19]:
from finances.present_value_and_cumulative_cashflows import cumulative_equity_and_debt_cashflows

df_cumulative_equity_and_debt_cashflows = cumulative_equity_and_debt_cashflows(df_equity_funding,
                                                                               df_debt_and_loan_part1,
                                                                               **ot_parameters)
df_cumulative_equity_and_debt_cashflows.style.format(precision=2)

cumulative equity and debt,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
cumulative_equity_cashflow,-1.39,-4.86,-9.81,-14.76,-19.71,-24.63,-29.55,-34.45,-39.34,-44.20,-49.04,-53.86,-56.59,-57.13,-55.41,-51.32,-44.79,-34.56,-21.67,-6.01,-5.16
cumulative_debt_cashflow,-0.00,-10.41,-9.99,-9.55,-9.07,-8.57,-8.02,-7.44,-6.82,-6.16,-5.45,-4.68,-3.87,-3.00,-2.07,-1.07,-0.00,-0.00,-0.00,-0.00,-0.00


Discounted cash flows for levelized cost

In [20]:
def discounted_cashflows_for_levelized_cost(df_construction_phase,
                                            df_operational_phase,
                                            df_decommissioning_phase,
                                            df_taxes_and_profits_part2,
                                            df_project_reserves,
                                            **kwargs):
    '''This function creates a dataframe that contains the discounted cashflows for levelized cost calculations'''

    # 1. Get the parameters we need
    wacc = kwargs['wacc']

    # 2. Get the timelines
    calendar_years = construct_calendar_year_list(**kwargs)
    business_case_years = construct_business_case_year_list(**kwargs)
    operational_years = construct_operations_years_list(**kwargs)

    # construct df
    row_names = [
        'capex_h2_boiler', 'capex_h2_station', 'capex_h2_pipeline',
        'avoided_capex_ng_boiler', 'h2_boiler_opex', 'network_costs_h2', 
        'purchasing_h2', 'purchasing_hwi', 'avoided_ng_boiler_opex',  
        'network_costs_ng', 'eu_carbon_permits', 'ng_costs',
        'ng_tax_costs', 'decommissioning', 'interest_costs', 
        'contingency', 'tax_expenses', 'total_h2_consumed',
    ]
    df_discounted_levelized = pd.DataFrame(0.0, index=row_names, columns=calendar_years)
    df_discounted_levelized.columns.name = "discounted cashflows for levelized cost"

    # 4. Set up the discount term and calculate levelized costs and revenues
    discount_term = (1 + wacc) ** np.array(business_case_years)

# LEVELIZED COSTS & AVOIDED COSTS

    df_discounted_levelized.loc['capex_h2_boiler'] = df_construction_phase.loc['capex_h2_boiler'] / discount_term
    df_discounted_levelized.loc['capex_h2_station'] = df_construction_phase.loc['capex_h2_station'] / discount_term
    df_discounted_levelized.loc['capex_h2_pipeline'] = df_construction_phase.loc['capex_h2_pipeline'] / discount_term
    df_discounted_levelized.loc['avoided_capex_ng_boiler'] = df_construction_phase.loc['avoided_capex_ng_boiler'] / discount_term
    df_discounted_levelized.loc['h2_boiler_opex'] = df_operational_phase.loc['h2_boiler_opex'] / discount_term
    df_discounted_levelized.loc['network_costs_h2'] = df_operational_phase.loc['network_costs_h2'] / discount_term
    df_discounted_levelized.loc['purchasing_h2'] = df_operational_phase.loc['purchasing_h2'] / discount_term
    df_discounted_levelized.loc['purchasing_hwi'] = df_operational_phase.loc['purchasing_hwi'] / discount_term
    df_discounted_levelized.loc['avoided_ng_boiler_opex'] = df_operational_phase.loc['avoided_ng_boiler_opex'] / discount_term
    df_discounted_levelized.loc['network_costs_ng'] = df_operational_phase.loc['network_costs_ng'] / discount_term
    df_discounted_levelized.loc['eu_carbon_permits'] = df_operational_phase.loc['eu_carbon_permits'] / discount_term
    df_discounted_levelized.loc['ng_costs'] = df_operational_phase.loc['ng_costs'] / discount_term
    df_discounted_levelized.loc['ng_tax_costs'] = df_operational_phase.loc['ng_tax_costs'] / discount_term
    df_discounted_levelized.loc['decommissioning'] = df_decommissioning_phase.loc['total_cashflow_decommissioning'] / discount_term
    df_discounted_levelized.loc['interest_costs'] = df_taxes_and_profits_part2.loc['interest_costs'] / discount_term

    # contingency (injection + divident return)
    df_discounted_levelized.loc['contingency'] = (
        (df_project_reserves.loc['contingency_injection'] 
         + df_project_reserves.loc['contingency_reserve_to_dividents']) / discount_term)

    # tax expenses
    df_discounted_levelized.loc['tax_expenses'] = df_taxes_and_profits_part2.loc['tax_expenses'] / discount_term


    # calculate discounted h2 consumed
    df_discounted_levelized.loc['total_h2_consumed'] = np.where(
        np.isin(calendar_years, operational_years),
        ot_h2_demand_mwh / discount_term,
        0.0
    )
    

    return df_discounted_levelized



In [21]:
df_discounted_cashflows_for_levelized_cost = discounted_cashflows_for_levelized_cost(df_construction_phase,
                                                                                     df_operational_phase,
                                                                                     df_decommissioning_phase,
                                                                                     df_taxes_and_profits_part2,
                                                                                     df_project_reserves,
                                                                                     **ot_parameters)
df_discounted_cashflows_for_levelized_cost.style.format(precision=2)

discounted cashflows for levelized cost,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
capex_h2_boiler,0.00,-3.42,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
capex_h2_station,0.00,-0.90,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
capex_h2_pipeline,0.00,-9.78,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
avoided_capex_ng_boiler,0.00,1.55,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
h2_boiler_opex,0.00,0.00,-0.10,-0.09,-0.09,-0.08,-0.07,-0.07,-0.06,-0.06,-0.05,-0.05,-0.05,-0.04,-0.04,-0.04,-0.03,-0.03,-0.03,-0.03,0.00
network_costs_h2,0.00,0.00,-6.62,-6.11,-5.64,-5.21,-4.81,-4.44,-4.10,-3.78,-3.49,-3.22,-2.97,-2.74,-2.53,-2.34,-2.16,-1.99,-1.84,-1.70,0.00
purchasing_h2,0.00,0.00,-11.89,-11.03,-10.23,-9.50,-8.81,-8.17,-7.58,-7.03,-6.53,-6.05,-5.39,-4.79,-4.26,-3.78,-3.34,-2.95,-2.60,-2.29,0.00
purchasing_hwi,0.00,0.00,-12.07,-11.14,-10.28,-9.49,-12.52,-11.56,-10.67,-9.85,-9.09,-8.39,-7.74,-7.15,-6.60,-6.09,-5.62,-5.19,-4.79,-4.42,0.00
avoided_ng_boiler_opex,0.00,0.00,0.06,0.05,0.05,0.05,0.04,0.04,0.04,0.03,0.03,0.03,0.03,0.02,0.02,0.02,0.02,0.02,0.02,0.02,0.00
network_costs_ng,0.00,0.00,1.69,1.56,1.44,1.33,1.23,1.14,1.05,0.97,0.89,0.82,0.76,0.70,0.65,0.60,0.55,0.51,0.47,0.43,0.00


Levelized Costs

In [22]:
def levelized_cost_and_revenues(capex_h2_station_variable,
                                capex_h2_pipeline_variable,
                                capex_h2_boiler_variable,
                                capex_ng_boiler_variable,
                                opex_h2_boiler_variable,
                                network_costs_h2_variable,
                                h2_price_variable,
                                hwi_price_variable,
                                opex_ng_boiler_variable,
                                network_costs_ng_variable,
                                carbon_permits_variable,
                                ng_price_variable,
                                ng_tax_variable,
                                inflation_variable,
                                loan_percentage_variable,
                                loan_interest_rate_variable,
                                income_tax_rate_variable,
                                wacc_variable,
                                lifetime_investment_variable):
    ''' This function creates a dataframe that contains the levelized costs and revenues'''

    # get df discounted cashflows for levelized cost
    df_discounted_cashflows_for_levelized_cost = discounted_cashflows_for_levelized_cost(capex_h2_station_variable,
                                                                                         capex_h2_pipeline_variable,
                                                                                         capex_h2_boiler_variable,
                                                                                         capex_ng_boiler_variable,
                                                                                         opex_h2_boiler_variable,
                                                                                         network_costs_h2_variable,
                                                                                         h2_price_variable,
                                                                                         hwi_price_variable,
                                                                                         opex_ng_boiler_variable,
                                                                                         network_costs_ng_variable,
                                                                                         carbon_permits_variable,
                                                                                         ng_price_variable,
                                                                                         ng_tax_variable,
                                                                                         inflation_variable,
                                                                                         loan_percentage_variable,
                                                                                         loan_interest_rate_variable,
                                                                                         income_tax_rate_variable,
                                                                                         wacc_variable,
                                                                                         lifetime_investment_variable)


    # construct df
    row_names_levelized_cost = df_discounted_cashflows_for_levelized_cost.index.tolist()
    row_names_levelized_cost.pop()

    df_levelized_cost = pd.DataFrame(0.0, index=row_names_levelized_cost, columns=['cost','avoided_cost'])
    df_levelized_cost.columns.name = "levelized cost calculations"



    #### function
    def levelized_cost():
        '''This function calculates the levelized costs and revenues'''

        total_electricity_produced_sum = (df_discounted_cashflows_for_levelized_cost
                                          .loc['total_h2_consumed'].sum())

        for x in row_names_levelized_cost:

            if df_discounted_cashflows_for_levelized_cost.loc[x].sum() > 0:

                df_levelized_cost.loc[x]['avoided_cost'] = (df_discounted_cashflows_for_levelized_cost
                                                            .loc[x].sum() * 1E6 / total_electricity_produced_sum)
            else:
                df_levelized_cost.loc[x]['cost'] = -(df_discounted_cashflows_for_levelized_cost
                                                    .loc[x].sum() * 1E6 / total_electricity_produced_sum)

        return df_levelized_cost

    # update df to that obtained from function
    df_levelized_cost = levelized_cost()

    #### function
    def cost_benefits():
        '''This function calculates the cost benefits'''

        if df_levelized_cost['cost'].sum() < df_levelized_cost['avoided_cost'].sum():
            profits = df_levelized_cost['avoided_cost'].sum() - df_levelized_cost['cost'].sum()

        else:
            profits = 0

        return profits
    
    # add profits to df
    df_levelized_cost.loc['cost_benefits'] = [0,0]
    df_levelized_cost.loc['cost_benefits']['cost'] = cost_benefits()

    #### function   
    def additional_cost_gap():
        '''This function calculates the additional cost gap'''

        if df_levelized_cost['cost'].sum() > df_levelized_cost['avoided_cost'].sum():
            gap = df_levelized_cost['cost'].sum() - df_levelized_cost['avoided_cost'].sum()

        else:
            gap = 0

        return gap
    
    # add unprofitable gap to df
    df_levelized_cost.loc['additional_cost_gap'] = [0,0]
    df_levelized_cost.loc['additional_cost_gap']['avoided_cost'] = additional_cost_gap()


    return df_levelized_cost


In [23]:
def levelized_cost_and_revenues(df_discounted_cashflows_for_levelized_cost, **kwargs):
    '''
    Calculates the levelized costs and avoided costs (revenues/savings) per unit of consumption (EUR/MWh).
    '''

    # 1. Get the row names, except for total_h2_consumed
    row_names_levelized_cost = df_discounted_cashflows_for_levelized_cost.index.tolist()
    row_names_levelized_cost.remove('total_h2_consumed')

    # 2. Construct df
    df_levelized_cost = pd.DataFrame(0.0, index=row_names_levelized_cost, columns=['cost','revenues'])
    df_levelized_cost.columns.name = "levelized cost calculations"

    # 3. Calculate total discounted hydrogen production
    total_h2_consumed_sum = df_discounted_cashflows_for_levelized_cost.loc['total_h2_consumed'].sum()

    # 4. Calculate the sum for every row
    row_sums = df_discounted_cashflows_for_levelized_cost.loc[row_names_levelized_cost].sum(axis=1)

    # 5. Arrange data over 'revenues' (if positive) and 'costs' (if negative)
    # use factor 1E6 because cost&revenues are in MEUR
    df_levelized_cost['revenues'] = np.where(row_sums>0, row_sums * 1E6 / total_h2_consumed_sum, 0.0)
    df_levelized_cost['cost'] = np.where(row_sums<0, -row_sums * 1E6 / total_h2_consumed_sum, 0.0)

    # 6. Calculate total costs and revenues
    total_costs = df_levelized_cost['cost'].sum()
    total_revenues = df_levelized_cost['revenues'].sum()

    # 7. Calculate profits or unprofitable gap
    df_levelized_cost.loc['profits'] = [0.0, 0.0]
    if total_revenues > total_costs:
        df_levelized_cost.loc['profits', 'cost'] = total_revenues - total_costs

    df_levelized_cost.loc['unprofitable_gap'] = [0.0, 0.0]
    if total_costs > total_revenues: 
        df_levelized_cost.loc['unprofitable_gap', 'revenues'] = total_costs - total_revenues


    return df_levelized_cost

In [24]:
df_levelized_cost_and_revenues = levelized_cost_and_revenues(df_discounted_cashflows_for_levelized_cost,
                                                             **ot_parameters)
df_levelized_cost_and_revenues.style.format(precision=2)

levelized cost calculations,cost,revenues
capex_h2_boiler,2.18,0.00
capex_h2_station,0.58,0.00
capex_h2_pipeline,6.25,0.00
avoided_capex_ng_boiler,0.00,0.99
h2_boiler_opex,0.64,0.00
network_costs_h2,41.94,0.00
purchasing_h2,74.21,0.00
purchasing_hwi,97.46,0.00
avoided_ng_boiler_opex,0.00,0.38
network_costs_ng,0.00,10.73


Project KPI's

In [25]:
from finances.kpis import project_kpi

df_project_kpi = project_kpi(df_present_value_cashflows,
                             df_taxes_and_profits_part2,
                             df_construction_phase,
                             **ot_parameters)

df_project_kpi.style.format(precision=2)

Project KPIs,Value,Unit
net_present_value,-30.74,MEUR
internal_rate_of_return,-0.02,%
return_on_investment,-36.78,%
payback_period,-48.94,years
discounted_return_on_investment,-244.26,%
discounted_payback_period,-12.48,years


Equity KPI's

In [26]:
from finances.kpis import equity_kpi

df_project_kpi = equity_kpi(df_equity_funding,
                            df_present_value_cashflows,
                             **ot_parameters)

df_project_kpi.style.format(precision=2)

Equity KPIs,Value,Unit
net_present_value,-23.10,MEUR
internal_rate_of_return,-0.84,%
return_of_investment,4.33,%
payback_period,415.35,years
discounted_return_on_investment,-695.45,%
discounted_payback_period,-3.02,years


Output KPI's

In [27]:
from finances.kpis import output_kpi

df_project_kpi = output_kpi(df_levelized_cost_and_revenues,
                            **ot_parameters)

df_project_kpi.style.format(precision=2)

Output KPIs,Value,Unit
levelized_cost,226.33,Eur/MWh
levelized_revenues,110.73,Eur/MWh
levelized_profits,-115.60,Eur/MWh
